## Cuando se pone difícil

En muchas situaciones los archivos pueden tener datos estructurados de alguna manera que no permiten directamente su lectura con `pandas`. 

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
smn_path = Path.cwd() / 'data' / 'smn'
pronostico_file= 'pronostico_5dias20260612.txt'
pronostico_path = smn_path / pronostico_file

In [3]:
pronostico = pd.read_csv(pronostico_path)
pronostico

,************************************************************************************************
Producto basado en un modelo de pronóstico numérico del tiempo,
por lo tanto puede diferir del pronostico emitido por el SMN,NaN
************************************************************************************************,NaN
AEROPARQUE,NaN
================================================================================================,NaN
...,...
16/JUN/2026 12Hs. 14.8 115 | 3 0.0,NaN
16/JUN/2026 15Hs. 17.1 139 | 8 0.0,NaN
16/JUN/2026 18Hs. 15.8 113 | 14 0.0,NaN
16/JUN/2026 21Hs. 12.4 72 | 14 0.0,NaN


Probemos usando `read_fwf`, que es capaz de leer tablas con columnas de ancho fijo:

In [4]:
pronostico = pd.read_fwf(pronostico_path,skiprows=5)
pronostico


,AEROPARQUE
0,==============================================...
1,FECHA * TEMPERATURA VIENTO ...
2,(DIR | KM/H)
3,==============================================...
4,12/JUN/2026 00Hs. 8.5 347 | 4 ...
...,...
5422,16/JUN/2026 12Hs. 14.8 115 | 3 ...
5423,16/JUN/2026 15Hs. 17.1 139 | 8 ...
5424,16/JUN/2026 18Hs. 15.8 113 | 14 ...
5425,16/JUN/2026 21Hs. 12.4 72 | 14 ...


Como se esperaba, no parece ser la solución porque la estructura del archivo no es estrictamente de columnas de ancho fijo, sino de secciones de columnas de ancho fijo. Entonces tenemos que preprocesar el archivo para llegar a obtener esas secciones:

In [5]:
with open(smn_path / 'pronostico_5dias20260612.txt', 'r') as file:
    content = file.read()

# Dividir el contenido en secciones por aeropuerto
sep =  "================================================================================================"
sections = content.split(sep)
print(f"Secciones encontradas: {len(sections)}")
for i, section in enumerate(sections[0:3]):  # Imprime solo las primeras 3 secciones para no saturar la salida
    print(f"\n--- Sección {i+1} ---")
    print(section[:500])  # Imprime los primeros 500 caracteres de cada sección

Secciones encontradas: 355

--- Sección 1 ---
************************************************************************************************
 Producto basado en un modelo de pronóstico numérico del tiempo, 
 por lo tanto puede diferir del pronostico emitido por el SMN
 ************************************************************************************************
 
 AEROPARQUE
 

--- Sección 2 ---

      FECHA *          TEMPERATURA      VIENTO      PRECIPITACION(mm)
                                     (DIR | KM/H)                        
 

--- Sección 3 ---

  12/JUN/2026 00Hs.         8.5       347 |   4         0.0 
  12/JUN/2026 03Hs.         7.7       336 |   4         0.0 
  12/JUN/2026 06Hs.         6.8       351 |   5         0.0 
  12/JUN/2026 09Hs.         8.0       344 |   5         0.0 
  12/JUN/2026 12Hs.        14.5       336 |   7         0.0 
  12/JUN/2026 15Hs.        16.4       342 |   6         0.0 
  12/JUN/2026 18Hs.        11.1       348 |   4         0.0 
  

In [6]:
print("Estación : ", sections[0])
print("Header : ", sections[1])
print("Pronóstico : ", sections[2])


Estación :  ************************************************************************************************
 Producto basado en un modelo de pronóstico numérico del tiempo, 
 por lo tanto puede diferir del pronostico emitido por el SMN
 ************************************************************************************************
 
 AEROPARQUE
 
Header :  
      FECHA *          TEMPERATURA      VIENTO      PRECIPITACION(mm)
                                     (DIR | KM/H)                        
 
Pronóstico :  
  12/JUN/2026 00Hs.         8.5       347 |   4         0.0 
  12/JUN/2026 03Hs.         7.7       336 |   4         0.0 
  12/JUN/2026 06Hs.         6.8       351 |   5         0.0 
  12/JUN/2026 09Hs.         8.0       344 |   5         0.0 
  12/JUN/2026 12Hs.        14.5       336 |   7         0.0 
  12/JUN/2026 15Hs.        16.4       342 |   6         0.0 
  12/JUN/2026 18Hs.        11.1       348 |   4         0.0 
  12/JUN/2026 21Hs.         9.8       359 |   7   

La lista `sections` contiene cada sección que corresponde a una estación meteorológica, cuyos nombres son:

In [7]:
print(sections[0].split('\n')[-2].strip()) # tratamiento especial para remover el comentario
print(sections[3].strip())
print(sections[6].strip())
print(sections[9].strip())

AEROPARQUE
AZUL_AERO
BAHIA_BLANCA_AERO
BARILOCHE_AERO


> La primer sección debe trabajarse a mano debido al comentario que posee el archivo

Las siguientes serían las secciones de headers que en principio no nos harían falta:

In [8]:
print(sections[1].strip())
print(sections[4].strip())
print(sections[7].strip())

FECHA *          TEMPERATURA      VIENTO      PRECIPITACION(mm)
                                     (DIR | KM/H)
FECHA *          TEMPERATURA      VIENTO      PRECIPITACION(mm)
                                     (DIR | KM/H)
FECHA *          TEMPERATURA      VIENTO      PRECIPITACION(mm)
                                     (DIR | KM/H)


Finalmente tenemos las secciones con los datos:

In [9]:
print(sections[2].split('\n')[:5])
print(sections[5].split('\n')[:5])
print(sections[8].split('\n')[:5])

['', '  12/JUN/2026 00Hs.         8.5       347 |   4         0.0 ', '  12/JUN/2026 03Hs.         7.7       336 |   4         0.0 ', '  12/JUN/2026 06Hs.         6.8       351 |   5         0.0 ', '  12/JUN/2026 09Hs.         8.0       344 |   5         0.0 ']
['', '  12/JUN/2026 00Hs.         7.6       331 |  11         0.0 ', '  12/JUN/2026 03Hs.         6.8       327 |  12         0.0 ', '  12/JUN/2026 06Hs.         5.9       329 |  12         0.0 ', '  12/JUN/2026 09Hs.         6.3       336 |  13         0.0 ']
['', '  12/JUN/2026 00Hs.        10.9       331 |  33         0.0 ', '  12/JUN/2026 03Hs.        11.1       330 |  36         0.0 ', '  12/JUN/2026 06Hs.        10.7       330 |  35         0.0 ', '  12/JUN/2026 09Hs.        10.9       330 |  37         0.0 ']


In [10]:
sections[2] 

'\n  12/JUN/2026 00Hs.         8.5       347 |   4         0.0 \n  12/JUN/2026 03Hs.         7.7       336 |   4         0.0 \n  12/JUN/2026 06Hs.         6.8       351 |   5         0.0 \n  12/JUN/2026 09Hs.         8.0       344 |   5         0.0 \n  12/JUN/2026 12Hs.        14.5       336 |   7         0.0 \n  12/JUN/2026 15Hs.        16.4       342 |   6         0.0 \n  12/JUN/2026 18Hs.        11.1       348 |   4         0.0 \n  12/JUN/2026 21Hs.         9.8       359 |   7         0.0 \n  13/JUN/2026 00Hs.        10.2       332 |   7         0.0 \n  13/JUN/2026 03Hs.         8.8       317 |   6         0.0 \n  13/JUN/2026 06Hs.         8.3       316 |   7         0.0 \n  13/JUN/2026 09Hs.         9.2       301 |   5         0.0 \n  13/JUN/2026 12Hs.        14.1       272 |   6         0.0 \n  13/JUN/2026 15Hs.        15.2       216 |   6         0.0 \n  13/JUN/2026 18Hs.        11.9       179 |  13         0.5 \n  13/JUN/2026 21Hs.         8.1       170 |  20         1.1 \n  14/

Ahora sí podemos usar `read_fwf` para transformar las secciones en `DataFrame`s. Para ello tenemos que convertir cada `section` (que es un string) en un tipo de buffer en memoria que se comporte como un archivo. Obsérvese que si hacemos:


In [12]:
from io import StringIO 

with StringIO(sections[5]) as data_io:
    df_malo = pd.read_fwf(data_io)
    
df_malo.head()

,12/JUN/2026,00Hs.,7.6,331,|,11,0.0
0,12/JUN/2026,03Hs.,6.8,327,|,12,0.0
1,12/JUN/2026,06Hs.,5.9,329,|,12,0.0
2,12/JUN/2026,09Hs.,6.3,336,|,13,0.0
3,12/JUN/2026,12Hs.,11.4,337,|,21,0.0
4,12/JUN/2026,15Hs.,14.0,328,|,20,0.0


Es decir que la primer fila del _stream_ es tomada como los nombres de las columnas. Tenemos que pasar el argumento opcional `names` para definir los nombres de las columnas:

In [13]:
from io import StringIO 

with StringIO(sections[2]) as data_io:
    df = pd.read_fwf(data_io,names=['dates','h','t','v_dir','l','v_vel','precip'])
    
df.head()

,dates,h,t,v_dir,l,v_vel,precip
0,12/JUN/2026,00Hs.,8.5,347,|,4,0.0
1,12/JUN/2026,03Hs.,7.7,336,|,4,0.0
2,12/JUN/2026,06Hs.,6.8,351,|,5,0.0
3,12/JUN/2026,09Hs.,8.0,344,|,5,0.0
4,12/JUN/2026,12Hs.,14.5,336,|,7,0.0



Para terminar podemos eliminar la columna `l` que no aporta información:

In [14]:
df.drop(columns=['l'],inplace=True)
df.head()

,dates,h,t,v_dir,v_vel,precip
0,12/JUN/2026,00Hs.,8.5,347,4,0.0
1,12/JUN/2026,03Hs.,7.7,336,4,0.0
2,12/JUN/2026,06Hs.,6.8,351,5,0.0
3,12/JUN/2026,09Hs.,8.0,344,5,0.0
4,12/JUN/2026,12Hs.,14.5,336,7,0.0


-----
## Ejercicio

- Procesar todas las estaciones
- Averiguar qué ropa me tengo que poner si quiero salir a festejar el triunfo de 🇦🇷 Argentina 🇦🇷 el martes ante Algeria.
  - Use una locación al azar si no está en Argentina.
  - La más cercana a su domicilio si está en Argentina.


### Una nota sobre el manejo de fechas...
Supongamos que usamos el formato 'dd/mm/aaaa' (13/06/2026) para la fecha, cómo llegamos a 'dd/MMM/aaaa' (13/JUN/2026)??

In [15]:
from datetime import datetime

datetime.strptime('15/JUN/2026', '%d/%b/%Y')

datetime.datetime(2026, 6, 15, 0, 0)

In [16]:
fecha = '13/06/2026'
# convertir a formato datetime sin usar pandas
fecha_dt = datetime.strptime(fecha, '%d/%m/%Y')
print(fecha_dt)
fecha_formateada = fecha_dt.strftime('%d/%b/%Y').upper()
print(fecha_formateada)


2026-06-13 00:00:00
13/JUN/2026


> OJO! Esto depende de 'locale'

In [17]:
fecha = '13/01/2026'
# convertir a formato datetime 
fecha_dt = datetime.strptime(fecha, '%d/%m/%Y')
print(fecha_dt)
fecha_formateada = fecha_dt.strftime('%d/%b/%Y').upper()
print(fecha_formateada)


2026-01-13 00:00:00
13/JAN/2026


In [18]:
import locale 

# Locale actual
locale.getlocale(locale.LC_TIME)

# Cambiar el locale a español de Argentina
locale.setlocale(locale.LC_TIME, "es_AR.UTF-8")

# convertir a formato datetime 
fecha_dt = datetime.strptime(fecha, '%d/%m/%Y')
print(fecha_dt)
fecha_formateada = fecha_dt.strftime('%d/%b/%Y').upper()
print(fecha_formateada)


2026-01-13 00:00:00
13/ENE./2026


Hay que parsear el '.' ese porque no está bien...

In [19]:
fecha_formateada = fecha_formateada.replace('.','')
print(fecha_formateada)

13/ENE/2026



## Ejercicio para el hogar

El archivo 'clima argentina 1981 2010.txt' contiene datos climáticos significativos de las distintas estaciones meteorológicas del Servicio Meteorológico Nacional.

- Inspeccione el archivo y diseñe un tipo `DataFrame` adecuado para contener dichos datos.
- Lea el archivo y cree el `DataFrame` que diseño en el item anterior.
- Cree una función para obtener el promedio anual de las magnitudes climáticas referidas en el archivo. La función debe recibir como argumentos el `DataFrame`, el nombre de la estación meteorológica y la magnitud, y devolver un valor (`float`) con el promedio.
- Genere un `DataFrame` que represente todas las magnitudes promedio para cada estación meteorológica, usando la función del item anterior.

-   Cree funciones para poder realizar gráficos comparativos de los datos meteorológicos. En particular, reproduzca

    - El gráfico de Valores Medios de Temperatura y Precipitación 
    - El gráfico de Valores extremos de Temperatura, usando la temperatura máxima y la mínima de cada mes.

que se pueden ver en el [sitio de estadísticas del Servicio Meteorológico Nacional](https://www.smn.gob.ar/estadisticas).

-----